# Phase 6 — Advanced Modeling

Searches the best (model × loss × label) combination per segment.

## Label Strategies Evaluated
| Strategy | Description |
|---|---|
| `raw` | Verified income as-is |
| `robust` | Winsorized at 2.5% tails |
| `quantile` | 70% own + 30% segment quantile (P25–P50 by segment) |
| `shrunk_composite` | `b1×median_credit + b2×stability − b3×volatility` + J-S shrinkage |

## Loss Functions Evaluated
`huber_10k` · `huber_20k` · `quantile_p30/40/50` · `log_rmse` · `mape` · `tweedie_p15`

## Models Evaluated
LightGBM × 8 losses · TabPFN v2 · (VanillaLSTM / BiLSTM / LSTMAttn / TCN if torch available)

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.modeling import LabelEngineer, SegmentModelTrainer
from src.modeling.loss_functions import LossRegistry, evaluate_income_predictions
from src.feature_selection import FeatureSelectionPipeline
from src.utils import setup_logging
from data.sample.generate_sample import generate_sample_training_data

setup_logging('INFO')

## Load Data + Feature Selection

In [ ]:
X_train, y_train = generate_sample_training_data(n=5000, seed=42)

# Feature selection
fs = FeatureSelectionPipeline(run_boruta=False)   # Fast mode
fs.fit(X_train, y_train, segment_col='segment')
print(f'Selected {len(fs.final_features_)} features')

## Label Engineering — Fit + Evaluate Strategies

In [ ]:
label_eng = LabelEngineer()
label_eng.fit(X_train, y_train, segment_col='segment')

# Compare all label strategies by CV MAE
eval_results = label_eng.evaluate(X_train, y_train, segment_col='segment', cv_folds=3)
print('Best strategy per segment:')
for seg, strat in label_eng.best_strategy_per_segment_.items():
    print(f'  {seg}: {strat}')

In [ ]:
# Heatmap: strategy × segment MAE
pivot = eval_results.pivot(index='strategy', columns='segment', values='cv_mae')

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot / 1000, annot=True, fmt='.1f', cmap='RdYlGn_r', ax=ax)
ax.set_title('CV MAE by Label Strategy × Segment (THB thousands)')
plt.tight_layout()
plt.show()

## Composite Label Decomposition

In [ ]:
# Show all labels side-by-side for a sample of customers
all_labels = label_eng.get_all_labels(X_train, y_train, segment_col='segment')
all_labels.head(10).round(0)

In [ ]:
# Composite coefficients by segment
print('Composite label coefficients per segment:')
for seg, coefs in label_eng.segment_coefs_.items():
    print(f'  {seg}: {coefs.round(3).to_dict()}')

## Loss Function Comparison (LightGBM)

In [ ]:
# Quick benchmark: all losses on full training set (no segmentation)
from sklearn.model_selection import cross_val_score, KFold
import lightgbm as lgb

feat_cols = [c for c in fs.final_features_ if c in X_train.columns]
X_feat = X_train[feat_cols].fillna(0)

print('Available losses:', LossRegistry.list_losses()['name'].tolist())

## Segment Model Trainer — Full Search

In [ ]:
trainer = SegmentModelTrainer(
    label_engineer=label_eng,
    feature_selector=fs,
    cv_folds=3,
    lgb_losses=['huber_10k', 'quantile_p40', 'log_rmse'],   # Subset for speed
    label_strategies=['raw', 'robust', 'shrunk_composite'],
    include_lstm=False,    # Set True if torch installed
    include_tabpfn=False,  # Set True if tabpfn installed
    max_rows_per_segment=2000,
)
trainer.fit(X_train, y_train, segment_col='segment')

In [ ]:
# Winner per segment
print('Best (model, loss, label) per segment:')
for seg, best in trainer.best_per_segment_.items():
    print(f"  {seg}: {best['model']} | {best['loss']} | {best['label']} | MAE={best['cv_mae']:,.0f}")

In [ ]:
# Full results table
results = trainer.results_summary()
results.groupby('segment').head(3).sort_values(['segment', 'cv_mae'])

## Error Analysis — Segment vs Baseline

In [ ]:
# Score on training data (in-sample — replace with test set in practice)
preds = trainer.predict(X_train, segment_col='segment')

eval_df = evaluate_income_predictions(
    y_train.reset_index(drop=True),
    preds.reset_index(drop=True),
    segment=X_train['segment'].reset_index(drop=True)
)
print(eval_df.to_string())

In [ ]:
# Error variance by segment — validates segmentation hypothesis
fig, ax = plt.subplots(figsize=(10, 5))
seg_mae = eval_df[eval_df.segment != 'ALL'].set_index('segment')
seg_mae[['mae', 'median_ae']].plot(kind='bar', ax=ax)
ax.set_title('MAE / MedAE by Segment (best model per segment)')
ax.set_ylabel('THB')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Save Artifacts

In [ ]:
label_eng.save('../artifacts/models/label_engineer.pkl')
trainer.save('../artifacts/models/segment_trainer/')
print('Saved.')